<a href="https://colab.research.google.com/github/annanya-sharma1002/AI_for_Health/blob/main/AI_for_health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages
from datetime import timedelta

def run_visualization(folder_path):
    p_id = os.path.basename(folder_path.rstrip('/'))

    # Load data
    nasal = pd.read_csv(f"{folder_path}/Flow.csv", parse_dates=['timestamp'])
    thoracic = pd.read_csv(f"{folder_path}/Thorac.csv", parse_dates=['timestamp'])
    spo2 = pd.read_csv(f"{folder_path}/SPO2.csv", parse_dates=['timestamp'])
    events = pd.read_csv(f"{folder_path}/Flow_Events.csv")
    events['start_time'] = pd.to_datetime(events['start_time'])

    events['end_time'] = pd.to_datetime(events['start_time'].dt.date.astype(str) + ' ' + events['end_time'])

    os.makedirs("Visualizations", exist_ok=True)
    save_path = f"Visualizations/{p_id}_Detailed_Report.pdf"

    # Start PDF context
    with PdfPages(save_path) as pdf:
        start_time = nasal['timestamp'].min()

        # Loop 96 times (96 * 5 minutes = 480 minutes / 8 hours)
        for i in range(96):
            window_start = start_time + timedelta(minutes=5 * i)
            window_end = window_start + timedelta(minutes=5)

            # Filter data for the current 5-minute window
            mask_n = (nasal['timestamp'] >= window_start) & (nasal['timestamp'] < window_end)
            mask_t = (thoracic['timestamp'] >= window_start) & (thoracic['timestamp'] < window_end)
            mask_s = (spo2['timestamp'] >= window_start) & (spo2['timestamp'] < window_end)

            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)

            # Plot segmented data
            ax1.plot(nasal.loc[mask_n, 'timestamp'], nasal.loc[mask_n, 'value'], color='#1f77b4', lw=0.7)
            ax2.plot(thoracic.loc[mask_t, 'timestamp'], thoracic.loc[mask_t, 'value'], color='#2ca02c', lw=0.7)
            ax3.plot(spo2.loc[mask_s, 'timestamp'], spo2.loc[mask_s, 'value'], color='#d62728', lw=1.5)

            # Overlay events that fall within this window
            window_events = events[(events['start_time'] < window_end) & (events['end_time'] > window_start)]
            for _, event in window_events.iterrows():
                for ax in [ax1, ax2, ax3]:
                    ax.axvspan(event['start_time'], event['end_time'], color='orange', alpha=0.3)

            # Styling and Labels
            ax1.set_title(f"Participant {p_id} | Page {i+1}/96 | Window: {window_start.strftime('%H:%M:%S')} - {window_end.strftime('%H:%M:%S')}")
            ax1.set_ylabel("Nasal")
            ax2.set_ylabel("Thoracic")
            ax3.set_ylabel("SpO2 %")

            plt.tight_layout()
            pdf.savefig(fig) # Save current page
            plt.close(fig)   # Close figure to free up memory

    print(f"Generated 96-page PDF: {save_path}")

if __name__ == "__main__":
    folder_path = "/content/drive/MyDrive/Data/AP01"
    run_visualization(folder_path)



/tmp/ipykernel_673/315577001.py:12: UserWarning: Parsing dates in %d.%m.%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  nasal = pd.read_csv(f"{folder_path}/Flow.csv", parse_dates=['timestamp'])
/tmp/ipykernel_673/315577001.py:13: UserWarning: Parsing dates in %d.%m.%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  thoracic = pd.read_csv(f"{folder_path}/Thorac.csv", parse_dates=['timestamp'])
/tmp/ipykernel_673/315577001.py:14: UserWarning: Parsing dates in %d.%m.%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  spo2 = pd.read_csv(f"{folder_path}/SPO2.csv", parse_dates=['timestamp'])
/tmp/ipykernel_673/315577001.py:16: UserWarning: Parsing dates in %d.%m.%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayf

Generated 96-page PDF: Visualizations/AP01_Detailed_Report.pdf
